# Notebook 00 — Configuración del Entorno en Google Colab

> **Blog Técnico Universitario** · *Traducción Bidireccional Imágenes Satelitales ↔ Bocetos Cartográficos*  
> Arquitectura: **U-Net 256 + PatchGAN 70×70** (Pix2Pix cGAN)

Este cuaderno prepara Google Colab para ejecutar todos los experimentos del proyecto. Cubre:

1. Verificar la GPU asignada (T4 gratuita, ~13 GB VRAM)
2. Montar Google Drive para persistir checkpoints entre sesiones
3. Configurar el repositorio y el `PYTHONPATH`
4. Instalar dependencias adicionales
5. Verificar que todos los módulos del proyecto se importan correctamente

---
**Referencias:**  
- Arquitectura Pix2Pix: https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix  
- Sketch2Map: https://github.com/PerlMonker303/S2MP

## 1. Verificación de GPU

Pix2Pix requiere GPU para entrenar en tiempo razonable. En la **T4 gratuita** de Colab:

| Parámetro | Valor |
|-----------|-------|
| VRAM disponible | ~13 GB |
| VRAM con `batch=1` + AMP | ~2–3 GB |
| Tiempo por época (1 096 pares) | ~2–4 min |
| Tiempo total (200 épocas) | ~8–13 horas |

> **Tip:** Colab desconecta la sesión tras ~12 h de inactividad. El script `train.py` guarda checkpoints cada 10 épocas para poder reanudar con `--reanudar`.

In [ ]:
import torch

print("=" * 55)
print("  Verificación de Hardware")
print("=" * 55)

if torch.cuda.is_available():
    nombre_gpu  = torch.cuda.get_device_name(0)
    vram_total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_libre  = (torch.cuda.get_device_properties(0).total_memory
                   - torch.cuda.memory_reserved(0)) / 1e9
    print(f"  GPU      : {nombre_gpu}")
    print(f"  VRAM     : {vram_total:.1f} GB total | {vram_libre:.1f} GB libre")
    print(f"  CUDA     : {torch.version.cuda}")
    print(f"  PyTorch  : {torch.__version__}")
    if vram_total < 8:
        print("\n  [!] Menos de 8 GB VRAM. Usa: batch_size=1, use_amp=True, grad_accum=4")
    else:
        print("\n  [OK] GPU suficiente para entrenamiento Pix2Pix completo.")
else:
    print("  [!] No se detectó GPU.")
    print("      Ir a: Entorno de ejecución > Cambiar tipo de entorno > GPU T4")

print("=" * 55)

## 2. Montar Google Drive

Drive persiste los checkpoints entre sesiones. Sin Drive, los pesos se pierden al reiniciar Colab.

```
Mi unidad/satelite-blog/
    checkpoints/sat2sketch/   ← pesos G y D, optimizadores, historia de pérdidas
    results/sat2sketch/       ← imágenes de muestra cada N épocas
    logs/                     ← archivos .log con pérdidas por batch
    data/processed/           ← dataset Maps (opcional, ~500 MB)
```

In [ ]:
import os
from pathlib import Path

USAR_DRIVE = True  # Cambiar a False para ejecución local sin Drive

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        RUTA_DRIVE = Path('/content/drive/MyDrive/satelite-blog')
        RUTA_DRIVE.mkdir(parents=True, exist_ok=True)
        print(f"[OK] Drive montado en: {RUTA_DRIVE}")
    except ImportError:
        print("[!] No estás en Colab. Drive no disponible.")
        RUTA_DRIVE = None
else:
    RUTA_DRIVE = None
    print("[Info] Drive desactivado. Checkpoints solo en sesión actual.")

## 3. Configurar el Repositorio

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar entorno ───────────────────────────────────────────────────────────
EN_COLAB = Path('/content').exists() and 'google.colab' in str(globals())

if EN_COLAB:
    PROYECTO = Path('/content/satelite-blog')
    # Opción A — clonar desde GitHub:
    # !git clone https://github.com/TU_USUARIO/satelite-blog.git {PROYECTO}
    # Opción B — copiar desde Drive:
    # !cp -r /content/drive/MyDrive/satelite-blog {PROYECTO}
    # Opción C — subir manualmente con el panel de archivos de Colab
else:
    # Ejecución local: el notebook está dentro del proyecto
    PROYECTO = Path('..').resolve()

os.chdir(PROYECTO)
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

print(f"Directorio de trabajo : {os.getcwd()}")
print(f"Python path           : {PROYECTO}")

## 4. Instalar Dependencias

In [ ]:
import subprocess, sys

# En Colab, PyTorch ya viene instalado.
# Solo necesitamos las dependencias extra del proyecto.
deps = [
    ("scikit-image", "skimage"),
    ("torchmetrics", "torchmetrics"),
    ("tqdm",         "tqdm"),
    ("Pillow",       "PIL"),
    ("matplotlib",   "matplotlib"),
]

print("Verificando dependencias...")
for paquete, modulo in deps:
    try:
        __import__(modulo)
        print(f"  [OK] {paquete}")
    except ImportError:
        print(f"  Instalando {paquete}...", end=" ")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', paquete, '-q'])
        print("OK")

print("\nTodas las dependencias listas.")

## 5. Verificar Importaciones del Proyecto

In [ ]:
modulos = [
    ("src.models.generator",         "GeneradorUNet"),
    ("src.models.discriminator",      "DiscriminadorPatchGAN"),
    ("src.models.losses",             "GANLoss"),
    ("src.data.dataset_loader",       "DatasetParesSideBySide"),
    ("src.training.config",           "ConfigEntrenamiento"),
    ("src.training.trainer",          "EntrenadorPix2Pix"),
    ("src.training.checkpointing",    "guardar_checkpoint"),
    ("src.utils.visualization",       "mostrar_grilla_muestras"),
    ("train",                         "bucle_entrenamiento"),
]

print("Verificando módulos del proyecto:")
todos_ok = True
for modulo, clase in modulos:
    try:
        obj = __import__(modulo, fromlist=[clase])
        getattr(obj, clase)
        print(f"  [OK] {modulo}.{clase}")
    except (ImportError, AttributeError) as e:
        print(f"  [!] {modulo}: {e}")
        todos_ok = False

if todos_ok:
    print("\n[OK] Entorno configurado correctamente.")
    print("     Siguiente notebook: 01_data_exploration.ipynb")
else:
    print("\n[!] Algunos módulos no se encontraron.")
    print("    Verifica que estás en el directorio raíz del proyecto.")